In [1]:
import dask
import time
import toolviper
import random
import webbrowser
import pathlib

import toolviper.utils.logger as logger

import toolviper.dask.client as client

from xradio.measurement_set import convert_msv2_to_processing_set
from xradio.measurement_set import open_processing_set

In [2]:
try:
    from prefect import task
    from prefect import flow

except ImportError:
    import sys
    import subprocess

    python = sys.executable
    result = subprocess.run(["python", "-m", "pip", "install", "prefect"], capture_output=True, text=True)
    logger.debug(f"Installation result: \n\t{result}")

finally:
    from prefect import task
    from prefect import flow
    


In [3]:
client = client.local_client(
    cores=12,
    log_params={
        "log_to_file": False, 
        "log_to_term": True, 
        "log_level": "DEBUG"
        },
    
    worker_log_params={
        "log_to_file": False, 
        "log_to_term": True, 
        "log_level": 
        "INFO"
        },
)

[2026-08-27 10:21:51,408]  WARNING      client:  It is recommended that the local cache directory be set using the dask_local_dir parameter. 


/opt/homebrew/Caskroom/mambaforge/base/envs/graphviper-i62/lib/python3.13/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 51301 instead
  warnings.warn(


[2026-08-27 10:21:52,336]    DEBUG      client:  Loading plugin module: <class 'worker.DaskWorker'>
[2026-08-27 10:21:53,794]     INFO      client:  Client <MenrvaClient: 'tcp://127.0.0.1:51302' processes=12 threads=12, memory=11.57 GiB> 


In [4]:
client.dashboard_link

'http://127.0.0.1:51301/status'

#### Utilities

In [5]:
def generate_delay(n=1, m=2):
    time.sleep(random.uniform(n, m))


def args_checker(args):
    if "delay" in args[0].keys():
        if args[0]["task_coords"]["antenna_name"]["data"] == args[0]["delay"]:
            return True

    return False


def write_image(file_name, spw, antenna, field):

    path = pathlib.Path(file_name)
    logger.info(str(path))
    if not path.exists():
        logger.error(f"File: {str(path)} not found.")

    data_path = path.joinpath(spw).joinpath(field).joinpath(antenna)
    logger.info(str(data_path))
    data_path.mkdir(exist_ok=True, parents=True)

    with open(
        str(data_path.joinpath(f"some_data_{spw}.{field}.{antenna}.image.ps.zarr")), "w"
    ) as file:
        file.write(f"{spw}\t{field}\t{antenna}")

def process_block(job, coordinates, functions, mode="single_node"):
    if not isinstance(functions, list):
        functions = [functions]
        
    for function in functions:
        job.make_coordinates(coords=coordinates)
        job.build_node(ps_partition=coordinates)
        
        job.map(
            function=function,
            parameters={}
        )
    
        job.reduce(
            function=gather,
            mode=mode,
        )
    
        job.compute()

### Make Test Functions 

In [6]:
UPPER_DELAY_LIMIT = 1


# Serial
@task
def Initialization(*args, **kwargs):
    logger.info("Initialization ... serial processing")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# Block 1

# Define image parameters #[field, spw, antenna]
@task
def define_image_parameters(*args, **kwargs):
    logger.info("Define image parameters... [field, spw, antenna]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)


# per-antenna stoke #[field, spw, antenna, scan, pol]
@task
def per_antenna_stokes(*args, **kwargs):
    logger.info("Per-antenna stokes ... [field, spw, antenna, scan, pol]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)


# per-antenna per polarization (XXYY) #[field, spw, antenna]
@task
def per_antenna_polarization_XXYY(*args, **kwargs):
    logger.info("Per-antenna per polarization XXYY ... [field, spw, antenna]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# Generate data array per antenna [field, spw, antenna]
@task
def generate_data_array(*args, **kwargs):
    logger.info("Generate data array for per antenna baseline rms map ... [field, spw, pol]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# Compute channel ranges #[field, spw, antenna]
@task
def heuristic_plots_per_eb(*args, **kwargs):
    logger.info("Pre-EB heuristic plotting ... [field, spw, antenna, pol]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

 ## End block 1

 # Block 2

# Gather per-antenna images per-field, per-spw [field, spw]
@task
def gather_per_antenna_processing(*args, **kwargs):
    logger.info("Gather per antenna images per-field, per-spw ... [field, spw, antenna]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# Combine per-antenna images [field, spw, antenna]
@task
def combine_per_antenna_stokes(*args, **kwargs):
    logger.info("Combine per antenna images ... [field, spw, antenna]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# Generate data array combine [field, spw, antenna]
@task
def generate_data_array_combine(*args, **kwargs):
    logger.info("Generate data array for combine baseline rms map ... [field, spw, pol]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# Compute image sensitivity [field, spw]
@task
def compute_image_sensitivity(*args, **kwargs):
    logger.info("Compute image sensitivity ... [field, spw]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# Combine per-antenna per-pol image cubes [field, spw, antenna]
@task
def create_image_cubes(*args, **kwargs):
    logger.info("Combine per-antenna, per-pol image cubes ... [field, spw, antenna]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# QA Ananlysis, plot generation for combine image cubes [field, spw]
@task
def qa_analysis(*args, **kwargs):
    logger.info("QA analysis, plot generation for combine image cubes ... [field, spw]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

## End block 2
# Block 3

# QA Ananlysis, plot generation for combine image cubes [field, spw]
@task
def qa_analysis_combine_images(*args, **kwargs):
    logger.info("QA analysis, plot generation for combine image cubes ... [field, spw]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# QA Ananlysis for per-antenna image cubes [field, spw, antenna]
@task
def qa_analysis_per_antenna_images(*args, **kwargs):
    logger.info("QA Ananlysis for per-antenna image cubes ... [field, spw, antenna]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# Generate plots [field, spw, antenna]
@task
def generate_plots(*args, **kwargs):
    logger.info("Generate plots ... [field, spw, antenna]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

## End block 3

# Weblog rendering [serial]
@task
def render_weblog(*args, **kwargs):
    logger.info("Rendering weblog ... [field, spw]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# Standard Gather
@task
def gather(*args, **kwargs):
    logger.info(" Gathering ...")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)



### Convert the MSv2 to a Processing Set and Open

In [7]:
toolviper.utils.data.download(file="uid___A002_Xe3a5fd_Xe38e.small.ms", folder="data")

[2026-08-27 10:21:53,825]    DEBUG      client:  Checking parameter values for cloudflare.download
[2026-08-27 10:21:53,830]    DEBUG      client:  Module path: /opt/homebrew/Caskroom/mambaforge/base/envs/graphviper-i62/lib/python3.13/site-packages/toolviper
[2026-08-27 10:21:53,831]    DEBUG      client:  Found configuration for cloudflare.download in: /opt/homebrew/Caskroom/mambaforge/base/envs/graphviper-i62/lib/python3.13/site-packages/toolviper
[2026-08-27 10:21:53,831]    DEBUG      client:  Parameter configuration file: /opt/homebrew/Caskroom/mambaforge/base/envs/graphviper-i62/lib/python3.13/site-packages/toolviper/config/cloudflare.param.json
[2026-08-27 10:21:53,832]     INFO      client:  Initializing download... 
[2026-08-27 10:21:53,834]     INFO      client:  File already exists: /Users/joshua/Development/graphviper-i62/docs/data/uid___A002_Xe3a5fd_Xe38e.small.ms 


In [8]:
file_path = pathlib.Path().cwd().joinpath("data/uid___A002_Xe3a5fd_Xe38e.small.ms")

In [9]:
processing_set = str(file_path) + ".ps.zarr"

In [10]:
convert_msv2_to_processing_set(
    in_file=str(file_path),
    out_file=processing_set,
    parallel_mode="partition",
    persistence_mode="w",
)

[2026-08-27 10:21:54,287]    DEBUG      client:  Loaded MAIN columns in 0.01s (2,076 unique MAIN rows)
[2026-08-27 10:21:54,292]    DEBUG      client:  SOURCE processing in 0.00s (added SOURCE_ID=True)
[2026-08-27 10:21:54,293]    DEBUG      client:  EPHEMERIS processing in 0.01s (added EPHEMERIS_ID=True)
[2026-08-27 10:21:54,294]    DEBUG      client:  STATE processing in 0.00s (OBS_MODE=True, SUB_SCAN_NUMBER=True)
[2026-08-27 10:21:54,294]     INFO      client:  Updated partition scheme used: ['DATA_DESC_ID', 'OBS_MODE', 'OBSERVATION_ID', 'EPHEMERIS_ID'] 
[2026-08-27 10:21:54,298]    DEBUG      client:  Partition build in 0.00s; total 20 partitions
[2026-08-27 10:21:54,298]    DEBUG      client:  Total create_partitions time: 0.02s
[2026-08-27 10:21:54,298]     INFO      client:  Selected 20 partitions out of 20 
[2026-08-27 10:21:54,299]     INFO      client:  OBSERVATION_ID [0], DDI [0], STATE [10], FIELD [1], SCAN [2, 4, 6, 8, 10, 12], EPHEMERIS [-1] 
[2026-08-27 10:21:54,299]    

2026-08-27 01:21:59	SEVERE	getcolshapestring::WEIGHT	Exception Reported: Table DataManager error: Invalid operation: TSM: no array in row 0 of column WEIGHT in /Users/joshua/Development/graphviper-i62/docs/data/uid___A002_Xe3a5fd_Xe38e.small.ms/table.f21
2026-08-27 01:21:59	SEVERE	getcolshapestring::WEIGHT	Exception Reported: Table DataManager error: Invalid operation: TSM: no array in row 474 of column WEIGHT in /Users/joshua/Development/graphviper-i62/docs/data/uid___A002_Xe3a5fd_Xe38e.small.ms/table.f21
2026-08-27 01:21:59	SEVERE	getcolshapestring::WEIGHT	Exception Reported: Table DataManager error: Invalid operation: TSM: no array in row 168 of column WEIGHT in /Users/joshua/Development/graphviper-i62/docs/data/uid___A002_Xe3a5fd_Xe38e.small.ms/table.f21
2026-08-27 01:21:59	SEVERE	getcolshapestring::WEIGHT	Exception Reported: Table DataManager error: Invalid operation: TSM: no array in row 63 of column WEIGHT in /Users/joshua/Development/graphviper-i62/docs/data/uid___A002_Xe3a5fd_

In [11]:
ps = open_processing_set(
    ps_store=str(processing_set), scan_intents=["OBSERVE_TARGET#ON_SOURCE"]
)

In [12]:
ps["uid___A002_Xe3a5fd_Xe38e.small_16"].xr_ms.get_partition_info()

{'antenna_name': ['PM01_T703', 'PM02_T701', 'PM04_T704'],
 'spectral_window_name': 'X1408602737#ALMA_RB_03#BB_1#SW-01#FULL_RES_0',
 'spectral_window_intents': ['UNSPECIFIED'],
 'field_name': ['Ridge-W1_2',
  'Ridge-W2_4',
  'Ridge-W3_6',
  'Ridge-W4_8',
  'Ridge-W5_10'],
 'polarization': ['XX', 'YY'],
 'scan_name': ['11', '3', '5', '7', '9'],
 'source_name': ['Ridge-W1_2',
  'Ridge-W2_4',
  'Ridge-W3_6',
  'Ridge-W4_8',
  'Ridge-W5_10'],
 'scan_intents': ['OBSERVE_TARGET#ON_SOURCE'],
 'line_name': ['CO_v=0_1-0(ID=3768098)'],
 'data_group_name': 'base'}

### Example of `hsd_imaging()` using GraphViper Framework

In [21]:
# Spawn dashboard window in a separate tab,
# comment out if you don't want this to spawn.
webbrowser.open(url=client.dashboard_link)

@flow(log_prints=True)
def hsd_imaging(processing_set, leaves=None):

    # Begin Preprocessing
    ps = open_processing_set(
        ps_store=str(processing_set), scan_intents=["OBSERVE_TARGET#ON_SOURCE"]
    )
    
    job = toolviper.utils.sd.graph.Graph.from_dataset(ps)

    # If you want to filter sub-dataset
    if leaves:
        job.filter(leaves=leaves)

    # Serial processing ... Initialization
    Initialization()

    # Block 1 processing
    process_block(
        job=job,
        coordinates=["field_name", "antenna_name"],
        functions=[
            define_image_parameters, 
            per_antenna_stokes, 
            per_antenna_polarization_XXYY,
            generate_data_array,
            heuristic_plots_per_eb
        ],
        mode="single_node"
    )

    # Block 2 processing
    process_block(
        job=job,
        coordinates=["field_name"],
        functions=[
            gather_per_antenna_processing,
            combine_per_antenna_stokes,
            generate_data_array_combine,
            compute_image_sensitivity,
            create_image_cubes,
            qa_analysis
        ],
        mode="single_node"
    )

    # Block 3 processing
    process_block(
        job=job,
        coordinates=["field_name", "antenna_name"],
        functions=[
            qa_analysis_combine_images,
            qa_analysis_per_antenna_images
        ],
        mode="single_node"
    )

    render_weblog()
    

In [ ]:
hsd_imaging(
    processing_set=processing_set,
    leaves=[
        "uid___A002_Xe3a5fd_Xe38e.small_16"
    ]
)

10:45:05.845 | INFO    | Flow run 'furry-chowchow' - Beginning flow run 'furry-chowchow' for flow 'hsd-imaging'

[2026-08-27 10:45:07,426]     INFO      client:  Creating graph from dataset ... 
[2026-08-27 10:45:07,426]     INFO      client:  Dropping nodes: {'uid___A002_Xe3a5fd_Xe38e.small_18', 'uid___A002_Xe3a5fd_Xe38e.small_19', 'uid___A002_Xe3a5fd_Xe38e.small_17'} 
[2026-08-27 10:45:07,441]     INFO      client:  Initialization ... serial processing 


10:45:08.454 | INFO    | Task run 'Initialization-0e1' - Finished in state Completed()

[2026-08-27 10:45:08,459]     INFO      client:  Making field coordinate ... 
[2026-08-27 10:45:08,464]     INFO      client:  Making antenna coordinate ... 
[2026-08-27 10:45:08,592]    DEBUG      client:  chunk_index: 0, (np.int64(0), np.int64(0))
[2026-08-27 10:45:08,592]    DEBUG      client:  chunk_index: 1, (np.int64(0), np.int64(1))
[2026-08-27 10:45:08,592]    DEBUG      client:  chunk_index: 2, (np.int64(0), np.int64(2))
[2026-08-27 10:45:08,593]    DEBUG      client:  chunk_index: 3, (np.int64(1), np.int64(0))
[2026-08-27 10:45:08,593]    DEBUG      client:  chunk_index: 4, (np.int64(1), np.int64(1))
[2026-08-27 10:45:08,593]    DEBUG      client:  chunk_index: 5, (np.int64(1), np.int64(2))
[2026-08-27 10:45:08,593]    DEBUG      client:  chunk_index: 6, (np.int64(2), np.int64(0))
[2026-08-27 10:45:08,594]    DEBUG      client:  chunk_index: 7, (np.int64(2), np.int64(1))
[2026-08-27 10:45:08,594]    DEBUG      client:  chunk_index: 8, (np.int64(2), np.int64(2))
[2026-08-27 10